# Milestone M2 — Audit Data, EDA, Uji Kebocoran & Model Baseline (Topik T3-A)
**Kelompok 6 — Deep Learning A (Sistem Informasi, UTM)**  
*Anggota:* Attala Alif Ramadhani Tri Hida (`230441100144`), Nafaul Hernanda Romadlona (`240441100125`), M. Rafly Kurniawan (`240441100086`), Naufal Husain (`240441100038`)


In [ ]:
# Clone repositori di Google Colab & pasang dependensi
import os, sys, subprocess

REPO_URL = "https://github.com/attaramadhani/UAS-DL-Kelompok6-T3A-WBC-DomainShift.git"
REPO_NAME = "UAS-DL-Kelompok6-T3A-WBC-DomainShift"

if "google.colab" in sys.modules and not os.path.exists(REPO_NAME):
    subprocess.run(["git", "clone", REPO_URL], check=True)
if os.path.exists(REPO_NAME):
    os.chdir(REPO_NAME)
sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 1. Unduh Dataset Resmi (Mendeley) & Jalankan Audit Data + Split Tetap (`seed=42`)


In [ ]:
from src.audit_eda import main as run_audit
run_audit()


## 2. Tampilkan Grafik EDA & Preview Macenko / Reinhard Stain Normalization


In [ ]:
from IPython.display import Image, display
for fig_name in ["fig1_sample_overlay_d1_d2.png", "fig2_eda_distributions.png", "fig3_stain_norm_preview.png"]:
    print("===", fig_name, "===")
    display(Image(filename=f"docs/figures/{fig_name}"))


## 3. Latih & Evaluasi 1 Model Baseline (U-Net ResNet-34: Train D1 -> Uji In-Domain D1 & Cross-Domain D2)


In [ ]:
from src.dataset import download_and_extract_zheng
from src.trainer import run_experiment

domain_dirs = download_and_extract_zheng("data")
split_csvs = {"D1": "splits/zheng_d1_split.csv", "D2": "splits/zheng_d2_split.csv"}

baseline_res = run_experiment(
    run_name="M2_Baseline_UNet_D1_to_D2",
    owner="Kelompok 6 (Baseline M2)",
    domain_dirs=domain_dirs,
    split_csvs=split_csvs,
    train_domain="D1",
    target_domain="D2",
    arch="unet",
    encoder_name="resnet34",
    loss_name="ce_dice",
    aug_mode="geom",
    seed=42,
    epochs=30,
    log_csv_path="logs/m2_baseline_runs.csv",
)
print("Ringkasan Baseline M2:", baseline_res["row"])
